In [1]:
from opt_targeted_transfers import BinaryRateTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = BinaryRateTargetedTransfers(c_bar=2.15, n_regressors=5)

In [4]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset)

Fitting conditional binary_rate improvement for transfer size 0.01


100%|██████████| 300/300 [00:03<00:00, 76.78it/s, val loss=0.672]


Fitting conditional binary_rate improvement for transfer size 0.545


100%|██████████| 300/300 [00:03<00:00, 75.82it/s, val loss=1]   


Fitting conditional binary_rate improvement for transfer size 1.08


100%|██████████| 300/300 [00:03<00:00, 84.08it/s, val loss=0.981]


Fitting conditional binary_rate improvement for transfer size 1.615


100%|██████████| 300/300 [00:03<00:00, 83.03it/s, val loss=0.888]


Fitting conditional binary_rate improvement for transfer size 2.15


100%|██████████| 300/300 [00:03<00:00, 85.34it/s, val loss=0.874]


In [5]:
# Precomputation for policy optimization step.
import numpy as np
budgets = np.linspace(0.05, 2.15, 10)
tt.get_opt_transfer_sizes_given_budget_grid(validation_dataset, budgets=budgets)

In [6]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=budgets[2])
assignments = tt.run_opt(test_covariate_dataset)

In [7]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.25219038377801084,
 'post_transfer_poverty_rate': 0.46025682122202033,
 'policy_cost_per_capita': 0.5165927711236471,
 'budget': 0.5166666666666667,
 'policy_type': 'binary_rate',
 'd': 2}

In [ ]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(budgets[-2])
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res


{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.0185169981589105,
 'post_transfer_poverty_rate': 0.029077563181581976,
 'policy_cost_per_capita': 1.91621885014867,
 'budget': 1.9166666666666667,
 'policy_type': 'binary_rate',
 'd': 2}

In [9]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, 
               test_dataset=test_dataset, 
               metrics=['post_transfer_poverty_rate', 'post_transfer_poverty_gap'], 
               budgets=budgets)

{'post_transfer_poverty_rate': {'auc': 0.5689156989677577,
  'results': [0.6290799816528524,
   0.6290799816528524,
   0.46025682122202033,
   0.45498572584971636,
   0.24759221554932748,
   0.2265941037385875,
   0.05145046769917962,
   0.024633268713555583,
   0.029077563181581976,
   0.0]},
 'post_transfer_poverty_gap': {'auc': 0.34757807243572303,
  'results': [0.5392908177172533,
   0.5392908177172533,
   0.25219038377801084,
   0.24666194416218465,
   0.07851376597246974,
   0.06274737714190243,
   0.01991172742870688,
   0.0021418872207480634,
   0.0185169981589105,
   0.0]}}